<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/tiny-transformer/blob/main/Mini_Transformer_Text_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import tensorflow as tf
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Prepare the data

In [6]:
sentences = ["the cat sat on the mat the dog sat on the rug cats love milk dogs love bones the cat loves milk"]
special_tokens = ["<PAD>", "<START>", "<END>"]

tokens = word_tokenize(sentences[0].lower())
vocab_words = special_tokens + list(set(tokens))
vocab = {w:i for i,w in enumerate(vocab_words)}
id2word = {i:w for w,i in vocab.items()}
vocab_size = len(vocab)

In [7]:
encoder_tokens = tokens
decoder_input_tokens = ["<START>"] + tokens
decoder_target_tokens = tokens + ["<END>"]

encoder_ids = tf.constant([vocab[w] for w in encoder_tokens], dtype=tf.int32)
decoder_input_ids = tf.constant([vocab[w] for w in decoder_input_tokens], dtype=tf.int32)
decoder_target_ids = tf.constant([vocab[w] for w in decoder_target_tokens], dtype=tf.int32)

Hyper parameters

In [8]:
embedding_dim = 64
ffn_dim = 128
learning_rate = 0.001
epochs = 300

Layers

In [9]:
embedding_layer = tf.keras.layers.Embedding(vocab_size, embedding_dim)
output_layer = tf.keras.layers.Dense(vocab_size)

layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
layernorm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

ffn = tf.keras.Sequential([
    tf.keras.layers.Dense(ffn_dim, activation='relu'),
    tf.keras.layers.Dense(embedding_dim)
])

# Define Q/K/V Dense layers outside tf.function
Q_enc_layer = tf.keras.layers.Dense(embedding_dim)
K_enc_layer = tf.keras.layers.Dense(embedding_dim)
V_enc_layer = tf.keras.layers.Dense(embedding_dim)

Q_dec_layer = tf.keras.layers.Dense(embedding_dim)
K_dec_layer = tf.keras.layers.Dense(embedding_dim)
V_dec_layer = tf.keras.layers.Dense(embedding_dim)

Q_cross_layer = tf.keras.layers.Dense(embedding_dim)
K_cross_layer = tf.keras.layers.Dense(embedding_dim)
V_cross_layer = tf.keras.layers.Dense(embedding_dim)

optimizer = tf.keras.optimizers.Adam(learning_rate)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


Positional Encoding

In [10]:
def positional_encoding_tf(seq_len, d_model):
    pos = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)
    i = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)
    angles = pos / tf.pow(10000.0, (2 * (i//2)) / tf.cast(d_model, tf.float32))
    pe = tf.where(tf.math.floormod(i, 2) == 0, tf.sin(angles), tf.cos(angles))
    return pe

Attention

In [11]:
def self_attention(x, Q_layer, K_layer, V_layer, mask=None):
    Q = Q_layer(x)
    K = K_layer(x)
    V = V_layer(x)
    scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(tf.cast(embedding_dim, tf.float32))
    if mask is not None:
        scores += mask * -1e9
    weights = tf.nn.softmax(scores, axis=-1)
    return tf.matmul(weights, V)

def cross_attention(query, key_value):
    Q = Q_cross_layer(query)
    K = K_cross_layer(key_value)
    V = V_cross_layer(key_value)
    scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(tf.cast(embedding_dim, tf.float32))
    weights = tf.nn.softmax(scores, axis=-1)
    return tf.matmul(weights, V)

Attention

In [12]:

@tf.function
def train_step(enc_ids, dec_in_ids, dec_out_ids):
    with tf.GradientTape() as tape:
        # Encoder
        enc_emb = embedding_layer(enc_ids) + positional_encoding_tf(tf.shape(enc_ids)[0], embedding_dim)
        enc_out = layernorm1(enc_emb + self_attention(enc_emb, Q_enc_layer, K_enc_layer, V_enc_layer))

        # Decoder
        dec_emb = embedding_layer(dec_in_ids) + positional_encoding_tf(tf.shape(dec_in_ids)[0], embedding_dim)
        seq_len = tf.shape(dec_in_ids)[0]
        mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        dec_out = layernorm2(dec_emb + self_attention(dec_emb, Q_dec_layer, K_dec_layer, V_dec_layer, mask))

        # Cross attention
        dec_out = layernorm3(dec_out + cross_attention(dec_out, enc_out))

        # Feed-forward + logits
        dec_out = ffn(dec_out)
        logits = output_layer(dec_out)

        # Loss
        loss = loss_fn(dec_out_ids, logits)

    trainable_vars = (embedding_layer.trainable_variables + output_layer.trainable_variables +
                      ffn.trainable_variables +
                      Q_enc_layer.trainable_variables + K_enc_layer.trainable_variables + V_enc_layer.trainable_variables +
                      Q_dec_layer.trainable_variables + K_dec_layer.trainable_variables + V_dec_layer.trainable_variables +
                      Q_cross_layer.trainable_variables + K_cross_layer.trainable_variables + V_cross_layer.trainable_variables)

    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    return loss

Training Loop

In [13]:
data = [(encoder_ids, decoder_input_ids, decoder_target_ids)]

for epoch in range(epochs):
    total_loss = 0
    for enc, din, dout in data:
        loss = train_step(enc, din, dout)
        total_loss += loss
    if epoch % 50 == 0:
        print(f"Epoch {epoch} Loss: {total_loss.numpy():.4f}")

Epoch 0 Loss: 3.9293
Epoch 50 Loss: 0.2487
Epoch 100 Loss: 0.0052
Epoch 150 Loss: 0.0020
Epoch 200 Loss: 0.0012
Epoch 250 Loss: 0.0008


In [14]:
def generate_text(seed_sentence, max_len=10):
    enc = [vocab[w] for w in word_tokenize(seed_sentence.lower())]
    enc_ids = tf.constant(enc, dtype=tf.int32)
    enc_emb = embedding_layer(enc_ids) + positional_encoding_tf(tf.shape(enc_ids)[0], embedding_dim)
    enc_out = layernorm1(enc_emb + self_attention(enc_emb, Q_enc_layer, K_enc_layer, V_enc_layer))

    generated = [vocab["<START>"]]

    for _ in range(max_len):
        dec_ids = tf.constant(generated, dtype=tf.int32)
        dec_emb = embedding_layer(dec_ids) + positional_encoding_tf(tf.shape(dec_ids)[0], embedding_dim)
        seq_len = tf.shape(dec_ids)[0]
        mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        dec_out = layernorm2(dec_emb + self_attention(dec_emb, Q_dec_layer, K_dec_layer, V_dec_layer, mask))
        dec_out = layernorm3(dec_out + cross_attention(dec_out, enc_out))
        dec_out = ffn(dec_out)
        logits = output_layer(dec_out)
        next_id = tf.argmax(logits[-1]).numpy()
        if next_id == vocab["<END>"]:
            break
        generated.append(next_id)

    return " ".join([id2word[i] for i in generated[1:]])

In [16]:
print(generate_text("the cat"))

sat sat sat on on on sat sat sat sat
